# Months-of-the-year irrep diagnostic (GPT-2 / Gemma 2 2B)

**Purpose.** This is the *gate*, not the experiment. It tells you, for a real LM's
month-token embedding and unembedding, which regime you are in:

- **Gate 1 (validation live?)** Does the k=1 Fourier mode — the "months on a circle"
  structure — sit significantly above the spectrum-preserving null? If yes, there is a
  genuine feature to validate.
- **Gate 2 (bite live?)** Is the matrix low-rank enough (2a), and does a *naive* readout
  flag a **wrong** mode (k>=2) as present where the spectrum-preserving null correctly
  rejects it (2b)? If yes, the confound bites on real weights.

Months under "next month" form **Z/12**. Its real isotypic decomposition is the real DFT:
k=0 trivial (rank 1), k=1..5 rotation modes (rank 2 each, **k=1 = the circle**),
k=6 alternating (rank 1). The projectors are fixed *a priori* by representation theory —
**no estimation, so no independence/double-dipping problem.** The closed-form Beta null and
detection floor transfer here because Z/12 is an exact finite group.

The kernel below is identical to the one validated on synthetic ground truth
(completeness/idempotency/rank to machine precision; Beta null matched to MC; the bite
mechanism reproduced). Read the verdicts off the final table.

In [ ]:
# Colab setup
!pip -q install transformers torch scipy numpy huggingface_hub
# For Gemma (gated): authenticate once. GPT-2 needs no auth.
# from huggingface_hub import login; login()  # uncomment for Gemma

## 1. Validated Z/12 kernel (identical to the locally-checked version)

In [ ]:
import numpy as np
from scipy import stats

N = 12

def build_projectors(n=N):
    j = np.arange(n); P = {}
    for k in range(0, n // 2 + 1):
        c = np.cos(2*np.pi*k*j/n); s = np.sin(2*np.pi*k*j/n)
        cols = [c] + ([s] if np.linalg.norm(s) > 1e-12 else [])
        B = np.stack(cols, axis=1); B, _ = np.linalg.qr(B)
        P[k] = B @ B.T
    return P

def ranks(P): return {k: int(round(np.trace(Pk))) for k, Pk in P.items()}

def raw_fractions(W, P):
    tot = np.linalg.norm(W, "fro")**2
    return {k: (np.linalg.norm(Pk @ W, "fro")**2)/tot for k, Pk in P.items()}

def isotropic_beta(k, P, d, n=N):
    r = int(round(np.trace(P[k])))
    return stats.beta(d*r/2.0, d*(n-r)/2.0)

def haar_orthogonal(n, rng):
    A = rng.standard_normal((n, n)); Q, R = np.linalg.qr(A)
    return Q * np.sign(np.diag(R))

def spectrum_preserving_p(W, P, B=2000, seed=0):
    rng = np.random.default_rng(seed); n = W.shape[0]
    obs = raw_fractions(W, P); ge = {k: 1 for k in P}
    for _ in range(B):
        Wb = haar_orthogonal(n, rng) @ W
        fb = raw_fractions(Wb, P)
        for k in P:
            if fb[k] >= obs[k]: ge[k] += 1
    return {k: ge[k]/(B+1) for k in P}

def effective_rank(W):
    s = np.linalg.svd(W, compute_uv=False); e = s**2
    return e.sum()/e.max(), (e.sum()**2)/(e**2).sum(), s

# self-check the gates before trusting anything downstream
P = build_projectors()
assert np.abs(sum(P.values()) - np.eye(12)).max() < 1e-12
assert ranks(P) == {0:1,1:2,2:2,3:2,4:2,5:2,6:1}
print("kernel gates pass:", ranks(P))

## 2. Load model and extract the 12 month-token rows

We pull **both** the input embedding and the output embedding (unembedding) and report
whether they are tied. The only model axis indexed by the group is the **vocabulary axis**
restricted to the month tokens — so the parameter object is the 12xd month submatrix of
the (un)embedding, exactly the row-indexed-by-group setup validated for S5.

**Tokenization is where the friction lives.** We try the leading-space, capitalised form
(" January", the in-running-text form) and take the single-token version where it exists.
Any month that is multi-token is flagged loudly; mean-pooling subwords distorts the
geometry, so prefer a model/word-form where months are single tokens.

In [ ]:
CONFIG = "gpt2"   # options here: "gpt2"  or  "google/gemma-2-2b"

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained(CONFIG)
model = AutoModelForCausalLM.from_pretrained(CONFIG, torch_dtype=torch.float32)
model.eval()

MONTHS = ["January","February","March","April","May","June",
          "July","August","September","October","November","December"]

def month_token_ids(tok):
    ids, flags = [], []
    for m in MONTHS:
        cand = tok(" " + m, add_special_tokens=False)["input_ids"]
        if len(cand) == 1:
            ids.append(cand[0]); flags.append(("single", cand))
        else:
            cand2 = tok(m, add_special_tokens=False)["input_ids"]
            if len(cand2) == 1:
                ids.append(cand2[0]); flags.append(("single-nospace", cand2))
            else:
                ids.append(cand[0]); flags.append(("MULTI->first", cand))
    return ids, flags

ids, flags = month_token_ids(tok)
print("token handling per month (calendar order):")
for m, f in zip(MONTHS, flags):
    print(f"  {m:>10}: {f[0]:>14}  ids={f[1]}")
multi = [m for m,f in zip(MONTHS,flags) if f[0].startswith("MULTI")]
if multi:
    print("\n  !! WARNING: multi-token months ->", multi, " (geometry may be distorted)")

E_in  = model.get_input_embeddings().weight.detach().float().numpy()
out   = model.get_output_embeddings()
E_out = out.weight.detach().float().numpy() if out is not None else E_in
tied  = np.shares_memory(E_in, E_out) or np.allclose(E_in, E_out)
print(f"\nembedding tied to unembedding: {tied}   |  d = {E_in.shape[1]}")

W_embed   = E_in[ids, :]      # 12 x d, rows = months in calendar order
W_unembed = E_out[ids, :]
print("W_embed", W_embed.shape, "| W_unembed", W_unembed.shape)

## 3. The diagnostic table

For each k we report the raw fraction and three verdicts:
- **unif?** f_k vs uniform share r_k/12 — the most naive readout.
- **iso?** f_k vs the isotropic Beta (1-alpha) quantile — a less-naive readout (Prop 2).
- **SP p / SP?** the spectrum-preserving p-value (Def 1) — the sound rule.

A **Bonferroni** column corrects for scanning all 7 modes (alpha/7), since per-mode tests
inflate the family-wise rate (we verified this: per-mode ~0.05, family-wise ~0.19, Bonferroni
restores ~0.05). Report the corrected verdict for any across-k claim.

**Modeling note on k=0.** The k=0 (all-ones) mode captures the component constant across
months — often a large "mean token" direction unrelated to month *structure*. The circular
claim lives in **k=1**. We show the full spectrum; consider the row-mean-centered version too
(removes k=0) if k=0 dominates for non-structural reasons.

In [ ]:
def diagnostic(W, name, B=2000, alpha=0.05, seed=0):
    P = build_projectors(); d = W.shape[1]
    f  = raw_fractions(W, P)
    sp = spectrum_preserving_p(W, P, B=B, seed=seed)
    stable, pr, s = effective_rank(W)
    nmodes = len(P)
    print(f"\n=== {name}   (d={d})  stable-rank={stable:.2f}  PR-rank={pr:.2f} ===")
    print(f"   top singular values: {np.round(s[:6],3)}")
    hdr = f"{'k':>2} {'r':>2} {'raw f_k':>8} {'null mu':>8} {'unif?':>6} {'iso?':>5} {'SP p':>7} {'SP?':>5} {'Bonf?':>6}"
    print(hdr); print('-'*len(hdr))
    rows = {}
    for k in P:
        r = ranks(P)[k]; mu = r/12
        unif = 'PRES' if f[k] > mu + 1e-9 else '.'
        iso  = 'PRES' if f[k] > isotropic_beta(k,P,d).ppf(1-alpha) else '.'
        spv  = 'PRES' if sp[k] < alpha else '.'
        bonf = 'PRES' if sp[k] < alpha/nmodes else '.'
        rows[k] = dict(f=f[k], mu=mu, unif=unif, iso=iso, sp=sp[k], spv=spv, bonf=bonf)
        print(f"{k:>2} {r:>2} {f[k]:>8.4f} {mu:>8.4f} {unif:>6} {iso:>5} {sp[k]:>7.4f} {spv:>5} {bonf:>6}")
    return rows, (stable, pr)

rows_e, er_e = diagnostic(W_embed,   "EMBEDDING")
rows_u, er_u = diagnostic(W_unembed, "UNEMBEDDING")

## 4. Read the gates

In [ ]:
def read_gates(rows, eff, name, alpha=0.05, nmodes=7):
    stable, pr = eff
    g1 = rows[1]['sp'] < alpha/nmodes          # k=1 significant (Bonferroni)
    wrong = [k for k in [2,3,4,5,6] ]
    # 2b: a wrong mode that a NAIVE rule flags present but SP correctly rejects
    bite_modes = [k for k in wrong
                  if (rows[k]['iso']=='PRES' or rows[k]['unif']=='PRES') and rows[k]['sp'] >= alpha]
    low_rank = stable < 6.0                      # 2a: effectively low rank on the 12-axis
    print(f"\n----- GATES: {name} -----")
    print(f"Gate 1 (k=1 circle present, Bonferroni): {'PASS' if g1 else 'FAIL'}  (SP p={rows[1]['sp']:.4f}, raw f={rows[1]['f']:.3f})")
    print(f"Gate 2a (low rank): {'PASS' if low_rank else 'FAIL'}  (stable-rank={stable:.2f})")
    print(f"Gate 2b (naive false-positive in a wrong mode, SP rejects): {'PASS' if bite_modes else 'FAIL'}", 
          f" modes={bite_modes}" if bite_modes else "")
    if g1 and bite_modes and low_rank:
        print("=> BEST REGIME: validation live AND bite fires on real weights.")
    elif g1 and not bite_modes:
        print("=> VALIDATION REGIME: circle confirmed, but energy too concentrated for a")
        print("   real-weights wrong-mode false positive. Use the constructed surrogate (cell 5).")
    elif not g1:
        print("=> Gate 1 FAILED: circular feature not clean in this checkpoint. Switch model/word-form.")
    return g1, bool(bite_modes), low_rank

read_gates(rows_e, er_e, "EMBEDDING")
read_gates(rows_u, er_u, "UNEMBEDDING")

## 5. Constructed bite at this model's spectrum (guaranteed fallback)

If Gate 2b does not fire on the real matrix (energy too concentrated), this gives you the
bite with controlled ground truth at the **real model's singular-value spectrum**: build a
*structureless* matrix with the same singular values but random orientation, and show the
naive readout false-positives while the spectrum-preserving null rejects. This is the
real-model-scale version of the S4/S5/S6 0.36 -> 0.005 result. It is weaker than a
real-weights false positive (the input is constructed), but it always works.

In [ ]:
def constructed_bite(W_real, n_trials=200, B=400, alpha=0.05, seed=0):
    P = build_projectors(); d = W_real.shape[1]
    s = np.linalg.svd(W_real, compute_uv=False)        # borrow the REAL spectrum
    rng = np.random.default_rng(seed)
    wrong = [2,3,4,5]
    fp_unif=fp_iso=fp_sp=fp_bonf=0
    for t in range(n_trials):
        U,_ = np.linalg.qr(rng.standard_normal((12,12)))
        V,_ = np.linalg.qr(rng.standard_normal((d,12)))
        W = (U * s) @ V.T                               # same spectrum, random orientation
        f = raw_fractions(W,P); sp = spectrum_preserving_p(W,P,B=B,seed=t)
        if any(f[k] > ranks(P)[k]/12 for k in wrong): fp_unif+=1
        if any(f[k] > isotropic_beta(k,P,d).ppf(1-alpha) for k in wrong): fp_iso+=1
        if any(sp[k] < alpha for k in wrong): fp_sp+=1
        if any(sp[k] < alpha/7 for k in wrong): fp_bonf+=1
    print(f"Constructed structureless inputs at the model's spectrum, FP rate in wrong modes (k>=2):")
    print(f"  uniform baseline     : {fp_unif/n_trials:.3f}")
    print(f"  isotropic Beta       : {fp_iso/n_trials:.3f}")
    print(f"  spectrum-preserving  : {fp_sp/n_trials:.3f}")
    print(f"  spectrum-pres + Bonf : {fp_bonf/n_trials:.3f}   <- the calibrated rule")

constructed_bite(W_embed)

## What to take away

1. **Gate 1 PASS** → you have a real-model validation: the calibrated test confirms the
   Engels circle is genuine and rotation-sensitive, not a rank artifact. Answers the
   "toy target" objection.
2. **Gate 2b PASS** → the confound bites on *actual trained weights* with representation
   theory as ground truth (a wrong mode the naive readout calls present, the calibrated
   test rejects). This is the result that makes reviewers nervous.
3. **Gate 2b FAIL but cell 5 works** → you still have the constructed bite at real-model
   scale. Weaker, but always available.

Report corrected (Bonferroni) p-values for any across-mode claim. Check both EMBEDDING and
UNEMBEDDING — a disagreement is itself interesting (cf. your S5 23/24 near-agreement).